# BME 2080 — Week 9 Meeting Deliverable Exercise
## Machine Learning for Cancer Drug Response Prediction

---

**Today's pipeline steps:**

| Step | Activity |
|------|----------|
| **Step 5: Select an algorithm** | Your team is assigned either Elastic Net or Random Forest |
| **Step 6: Train and test** | Run the optimized model on all 15 drugs |
| **Step 7: Evaluate performance** | Interpret CV R² — which drugs are predictable? |
| **Step 8: Tune the model** | Sweep one hyperparameter and record results in Google Sheets |

**How to navigate:** Use the table of contents in the left sidebar to jump to your section.
- Everyone runs **Section 1** (setup)
- Elastic Net teams (EVEN) run **Section 2**
- Random Forest teams (ODD) run **Section 3**

---


## Section 1: Setup — Everyone Runs This


In [ ]:
# Block 1: Import all libraries needed for today
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

print('Libraries loaded successfully!')


In [ ]:
# Block 2: Load the dataset from GitHub
# 212 cancer cell lines x 143 molecular features + 15 drug response values

url = 'https://raw.githubusercontent.com/cr546-collab/bme2080-module3/main/teaching_multiomics_AUC_212lines.csv'
df = pd.read_csv(url)

# Features: RNA expression, protein expression, mutation status, cancer type
feature_cols = ([c for c in df.columns if c.startswith('exp_')] +
                [c for c in df.columns if c.startswith('prot_')] +
                [c for c in df.columns if c.startswith('mut_')] +
                [c for c in df.columns if c.startswith('lineage_')])

# Drug response columns (what we're predicting)
drug_cols = sorted([c for c in df.columns if c.startswith('drug_')])

print(f'Dataset loaded: {df.shape[0]} cell lines')
print(f'Features: {len(feature_cols)} '
      f'(RNA: {sum(1 for c in feature_cols if c.startswith("exp_"))}, '
      f'Protein: {sum(1 for c in feature_cols if c.startswith("prot_"))}, '
      f'Mutation: {sum(1 for c in feature_cols if c.startswith("mut_"))}, '
      f'Cancer type: {sum(1 for c in feature_cols if c.startswith("lineage_"))})')
print(f'Drugs: {len(drug_cols)}')


In [ ]:
# Block 3: Shared settings used by both teams

# 5-fold cross-validation: trains on 80% of data, tests on 20%, repeated 5 times
# The reported R² is the average across all 5 test sets
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cancer type colors — consistent across all plots
cancer_colors = {
    'Breast':                        '#e63946',
    'Colorectal':                    '#2a9d8f',
    'Lung Adenocarcinoma':           '#457b9d',
    'Lung Squamous Cell Carcinoma':  '#fb8500',
    'Ovary':                         '#8338ec',
    'Pancreas':                      '#06d6a0',
    'Skin':                          '#ffb703',
}

# The 4 drugs where the model finds meaningful signal
top4 = ['drug_Dabrafenib', 'drug_Vemurafenib', 'drug_Trametinib', 'drug_Afatinib']
top4_names = [d.replace('drug_', '') for d in top4]

print('Settings ready.')


---
## Section 2a (EVEN): Test and Train Elastic Net

> **Only run this section if your team is assigned Elastic Net.**
> Random Forest teams: use the table of contents to jump to Section 3.

---

### About Elastic Net

Elastic Net is a **linear regression model** with regularization. It learns a coefficient
for each feature — negative coefficients predict sensitivity (lower AUC), positive
coefficients predict resistance (higher AUC). Regularization automatically shrinks
unimportant coefficients to zero, selecting only the most useful features.

- ✅ Handles many correlated features well
- ✅ Interpretable coefficients
- ⚠️ Requires feature scaling before fitting
- ⚠️ Assumes linear relationships

We use **`ElasticNetCV`** which automatically finds the best regularization settings.
In **Section 3a**, you'll explore what happens when you change those settings manually.


In [ ]:
# ── Block 4a: Steps 6 & 7: Train and evaluate Elastic Net on all 15 drugs ─────────────
# This may take 1-2 minutes. Watch the R² values appear as each drug finishes.
#
# R² interpretation:
#   R² = 1.0 → perfect predictions
#   R² = 0.0 → model is no better than predicting the mean AUC for every cell line
#   R² < 0.0 → model is worse than predicting the mean

en_results = {}

for drug in drug_cols:
    name = drug.replace('drug_', '')
    valid = df[df[drug].notna()].reset_index(drop=True)
    y = valid[drug].values
    X = SimpleImputer(strategy='mean').fit_transform(valid[feature_cols])

    scores = []
    preds = np.zeros(len(y))
    best_alphas, best_l1s = [], []

    for tr, te in kf.split(X):
        sc = StandardScaler()  # scale features — required for Elastic Net
        Xtr = sc.fit_transform(X[tr])  # fit scaler on training data only
        Xte = sc.transform(X[te])      # apply same scale to test data
        m = ElasticNetCV(
            l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
            alphas=np.logspace(-3, 2, 50),
            cv=5, max_iter=5000, random_state=42
        )
        m.fit(Xtr, y[tr])
        preds[te] = m.predict(Xte)
        scores.append(r2_score(y[te], preds[te]))
        best_alphas.append(m.alpha_)
        best_l1s.append(m.l1_ratio_)

    en_results[name] = {
        'r2': np.mean(scores), 'preds': preds,
        'y': y, 'cancers': valid['disease'].values,
        'best_alpha': np.mean(best_alphas),
        'best_l1': np.mean(best_l1s)
    }
    print(f'{name:<20} CV R² = {np.mean(scores):>6.3f}')

print('\nDone! Which drugs show meaningful signal (R² > 0.2)?')


In [ ]:
# ── Block 5a: Plot 1: CV R² for all 15 drugs ──────────────────────────────────────────
# Orange bars = the 4 drugs where the model finds real signal
# Gray bars = drugs the model cannot predict
# Dashed line = R²=0.2 threshold for meaningful signal

drugs = list(en_results.keys())
r2_vals = [en_results[d]['r2'] for d in drugs]
bar_colors = ['#ff9f43' if d in top4_names else '#555555' for d in drugs]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(drugs, r2_vals, color=bar_colors, edgecolor='white', alpha=0.9)
ax.axhline(0, color='black', lw=1)
ax.axhline(0.2, color='gray', lw=1.5, linestyle='--', alpha=0.6,
           label='R²=0.2 threshold')
ax.set_ylabel('5-Fold CV R²', fontsize=12)
ax.set_title('Elastic Net: CV R² Across All 15 Drugs\n'
             '(orange = top 4 predictable drugs)',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(drugs, rotation=35, ha='right', fontsize=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Label the top 4 bars
for x, (drug, val) in enumerate(zip(drugs, r2_vals)):
    if drug in top4_names:
        ax.text(x, val + 0.008, f'{val:.2f}', ha='center',
                fontsize=9, fontweight='bold', color='#ff9f43')

plt.tight_layout()
plt.show()


In [ ]:
# ── Block 6a: Plot 2: Predicted vs. Actual AUC — top 4 drugs ──────────────────────────
# Each dot = one cell line, colored by cancer type
# Dashed diagonal = perfect prediction line
# Dots ON the diagonal = correct predictions
# Dots forming a horizontal cluster = model has no signal for that group

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('Elastic Net: Predicted vs. Actual AUC — Top 4 Drugs\n'
             '(dashed line = perfect prediction, colored by cancer type)',
             fontsize=13, fontweight='bold')

for ax, drug_name in zip(axes, top4_names):
    r = en_results[drug_name]
    for cancer in np.unique(r['cancers']):
        idx = np.where(r['cancers'] == cancer)[0]
        ax.scatter(r['y'][idx], r['preds'][idx],
                   color=cancer_colors[cancer], alpha=0.85,
                   edgecolors='white', linewidths=0.4, s=55, label=cancer)
    mn = min(r['y'].min(), r['preds'].min()) - 0.03
    mx = max(r['y'].max(), r['preds'].max()) + 0.03
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, alpha=0.5)
    rmse = np.sqrt(np.mean((r['y'] - r['preds'])**2))
    ax.text(0.04, 0.96,
            f'CV R² = {r["r2"]:.3f}\nRMSE = {rmse:.3f}\nn = {len(r["y"])}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_title(drug_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual AUC', fontsize=10)
    ax.set_ylabel('Predicted AUC', fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)

handles = [mpatches.Patch(color=v, label=k) for k, v in cancer_colors.items()]
fig.legend(handles=handles, loc='lower center', ncol=7,
           fontsize=9, bbox_to_anchor=(0.5, -0.06), framealpha=0.9)
plt.tight_layout()
plt.show()


---
## Section 3a (EVEN): Tune Elastic Net

Now you'll explore how **alpha** (regularization strength) affects model performance
for Dabrafenib — our most predictable drug.

- **Too low alpha** → model overfits → poor CV R²
- **Too high alpha** → model is too simple → also poor CV R²
- **Sweet spot** → best CV R²

**Instructions:**
1. Change the `alpha` value in the cell below
2. Run the cell
3. Record your result in the shared Google Sheet
4. Repeat with a new value

**Suggested values:** `0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0`


In [ ]:
# Block 7a
# ┌──────────────────────────────────────────────────┐
# │  >>> CHANGE THESE VALUES, THEN RUN THE CELL <<<  │
# └──────────────────────────────────────────────────┘

team_number = 2   # >>> SET THIS TO YOUR TEAM NUMBER (1-7) — do this first!
alpha = 0.001    # try: 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0

# ── Everything below runs automatically ──────────────────────────────────
kf_team = KFold(n_splits=5, shuffle=True, random_state=team_number)

print(f'Team {team_number} | alpha = {alpha}\n')
print(f'{"Drug":<15} {"CV R²":>8}')
print('-' * 25)

for drug in top4:
    name = drug.replace('drug_', '')
    valid = df[df[drug].notna()].reset_index(drop=True)
    y = valid[drug].values
    X = SimpleImputer(strategy='mean').fit_transform(valid[feature_cols])
    scores = []
    for tr, te in kf_team.split(X):
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        m = ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=5000)
        m.fit(Xtr, y[tr])
        scores.append(r2_score(y[te], m.predict(Xte)))
    print(f'{name:<15} {np.mean(scores):>8.4f}')

print(f'\n→ Record all 4 values in the Google Sheet, then change alpha and run again!')


---
## Section 2b (ODD): Test and Train Random Forest

> **Only run this section if your team is assigned Random Forest.**
> Elastic Net teams: you should already be done with Section 2.

---

### About Random Forest

Random Forest builds many decision trees and averages their predictions.
Each tree is trained on a random subset of the data using a random subset
of features at each split — this randomness makes the ensemble robust.

- ✅ No feature scaling required
- ✅ Handles many features well
- ✅ Provides feature importance scores
- ⚠️ Less interpretable than Elastic Net
- ⚠️ Can be slow with many trees

In **Section 3b**, you'll explore what happens when you change the number of trees.


In [ ]:
# ── Block 4b: Steps 6 & 7: Train and evaluate Random Forest on all 15 drugs ───────────
# This may take 2-3 minutes. Watch the R² values appear as each drug finishes.
#
# Note: no scaling needed for Random Forest — features go in raw

rf_results = {}

for drug in drug_cols:
    name = drug.replace('drug_', '')
    valid = df[df[drug].notna()].reset_index(drop=True)
    y = valid[drug].values
    X = SimpleImputer(strategy='mean').fit_transform(valid[feature_cols])

    scores = []
    preds = np.zeros(len(y))

    for tr, te in kf.split(X):
        m = RandomForestRegressor(
            n_estimators=300,      # 300 trees
            max_features='sqrt',   # consider sqrt(143) ≈ 12 features per split
            min_samples_leaf=3,    # each leaf needs at least 3 samples
            random_state=42,
            n_jobs=-1
        )
        m.fit(X[tr], y[tr])
        preds[te] = m.predict(X[te])
        scores.append(r2_score(y[te], preds[te]))

    rf_results[name] = {
        'r2': np.mean(scores), 'preds': preds,
        'y': y, 'cancers': valid['disease'].values
    }
    print(f'{name:<20} CV R² = {np.mean(scores):>6.3f}')

print('\nDone! Which drugs show meaningful signal (R² > 0.2)?')


In [ ]:
# ── Block 5b: Plot 1: CV R² for all 15 drugs ──────────────────────────────────────────

drugs_rf = list(rf_results.keys())
r2_vals_rf = [rf_results[d]['r2'] for d in drugs_rf]
bar_colors_rf = ['#00e5ff' if d in top4_names else '#555555' for d in drugs_rf]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(drugs_rf, r2_vals_rf, color=bar_colors_rf, edgecolor='white', alpha=0.9)
ax.axhline(0, color='black', lw=1)
ax.axhline(0.2, color='gray', lw=1.5, linestyle='--', alpha=0.6,
           label='R²=0.2 threshold')
ax.set_ylabel('5-Fold CV R²', fontsize=12)
ax.set_title('Random Forest: CV R² Across All 15 Drugs\n'
             '(blue = top 4 predictable drugs)',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(drugs_rf, rotation=35, ha='right', fontsize=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

for x, (drug, val) in enumerate(zip(drugs_rf, r2_vals_rf)):
    if drug in top4_names:
        ax.text(x, val + 0.008, f'{val:.2f}', ha='center',
                fontsize=9, fontweight='bold', color='#00e5ff')

plt.tight_layout()
plt.show()


In [ ]:
# ── Block 6b: Plot 2: Predicted vs. Actual AUC — top 4 drugs ──────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('Random Forest: Predicted vs. Actual AUC — Top 4 Drugs\n'
             '(dashed line = perfect prediction, colored by cancer type)',
             fontsize=13, fontweight='bold')

for ax, drug_name in zip(axes, top4_names):
    r = rf_results[drug_name]
    for cancer in np.unique(r['cancers']):
        idx = np.where(r['cancers'] == cancer)[0]
        ax.scatter(r['y'][idx], r['preds'][idx],
                   color=cancer_colors[cancer], alpha=0.85,
                   edgecolors='white', linewidths=0.4, s=55, label=cancer)
    mn = min(r['y'].min(), r['preds'].min()) - 0.03
    mx = max(r['y'].max(), r['preds'].max()) + 0.03
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, alpha=0.5)
    rmse = np.sqrt(np.mean((r['y'] - r['preds'])**2))
    ax.text(0.04, 0.96,
            f'CV R² = {r["r2"]:.3f}\nRMSE = {rmse:.3f}\nn = {len(r["y"])}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_title(drug_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual AUC', fontsize=10)
    ax.set_ylabel('Predicted AUC', fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)

handles = [mpatches.Patch(color=v, label=k) for k, v in cancer_colors.items()]
fig.legend(handles=handles, loc='lower center', ncol=7,
           fontsize=9, bbox_to_anchor=(0.5, -0.06), framealpha=0.9)
plt.tight_layout()
plt.show()


---
## Section 3b (ODD): Tune Random Forest

Now you'll explore how **n_estimators** (number of trees) affects performance
for Dabrafenib.

- **Too few trees** → unstable predictions, vary between runs
- **More trees** → more stable, but diminishing returns
- **Very many trees** → performance plateaus, compute time keeps increasing

**Instructions:**
1. Change the `n_estimators` value in the cell below
2. Run the cell
3. Record your result in the shared Google Sheet
4. Repeat with a new value

**Suggested values:** `1, 5, 10, 25, 50, 100, 200, 300, 500`

> Note: small values run fast (seconds). Large values (300-500) take ~30 seconds.


In [ ]:
# Block 7b:
# ┌──────────────────────────────────────────────────┐
# │  >>> CHANGE THESE VALUES, THEN RUN THE CELL <<<  │
# └──────────────────────────────────────────────────┘

team_number = 1    # >>> SET THIS TO YOUR TEAM NUMBER (1-7) — do this first!
n_estimators = 1 # try: 1, 5, 10, 25, 50, 100, 200, 300, 500

# ── Everything below runs automatically ──────────────────────────────────
kf_team = KFold(n_splits=5, shuffle=True, random_state=team_number)

print(f'Team {team_number} | n_estimators = {n_estimators}\n')
print(f'{"Drug":<15} {"CV R²":>8}')
print('-' * 25)

for drug in top4:
    name = drug.replace('drug_', '')
    valid = df[df[drug].notna()].reset_index(drop=True)
    y = valid[drug].values
    X = SimpleImputer(strategy='mean').fit_transform(valid[feature_cols])
    scores = []
    for tr, te in kf_team.split(X):
        m = RandomForestRegressor(
            n_estimators=n_estimators,
            max_features='sqrt',
            min_samples_leaf=3,
            random_state=team_number,
            n_jobs=-1
        )
        m.fit(X[tr], y[tr])
        scores.append(r2_score(y[te], m.predict(X[te])))
    print(f'{name:<15} {np.mean(scores):>8.4f}')

print(f'\n→ Record all 4 values in the Google Sheet, then change n_estimators and run again!')
